# 01 - Data Understanding

## Objective

This step documents the dataset structure before exploratory analysis, cleaning, and modeling. It examines dimensions, data types, missing values, initial distributions, descriptive statistics, inconsistencies, and the target variable.

The main questions are:

- How many observations and variables are available?
- Which variables are numeric or categorical?
- Are there missing or redundant variables?
- Which target should be used for forecasting?

In [ ]:
from pathlib import Path
import pandas as pd

# Caminho of the project
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "raw"

# Load the data
csv_file = DATA_PATH / "data.csv"
df = pd.read_csv(csv_file)

In [ ]:
# dimension of the dataset
n_samples, n_features = df.shape

print("=" * 35)
print("DATASET DIMENSIONS")
print("=" * 35)
print(f"Number of observations : {n_samples}")
print(f"Number of features   : {n_features}")

In [ ]:
df.head()

## Dataset Structure

### Missing Values and Variable Types

In [ ]:
print("information about the dataset:")
df.info()

The initial dataset contains 20,805 observations and nine features. The original Portuguese column names are preserved because they are part of the data contract. The fields represent product identification, description, prices, quantity, sale date, cost, balance, and product group.

In [ ]:
# missing values
print("missing values by column:")
print(df.isna().sum())

In [ ]:
print("number of rows duplicadas:")
print(df.duplicated().sum())

In [ ]:
df[df.duplicated()]

These rows are not true duplicates: they represent valid sales with different order or product identifiers.

## Initial Hypotheses

Business understanding and the dataset structure suggest that revenue forecasting is more reliable than profit forecasting because cost and balance fields contain many missing values. The analysis should also account for products sold by both unit and length.

---

## Data Preprocessing

### Name and Variable-Type Processing

In [ ]:
banco_inicial = df.copy()

banco_inicial.rename(columns={'produto': 'id_produto'}, inplace=True)

In [ ]:
# Convert monetary columns to numeric values

colunas_monetarias = [
    'valor_unitario',
    'valor_total',
    'preco_custo',
    'saldo'
]

for col in colunas_monetarias:
    banco_inicial[col] = (
        banco_inicial[col]
        .astype(str)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
    )

    banco_inicial[col] = pd.to_numeric(
            banco_inicial[col],
            errors='coerce' # Convert non-numeric values to NaN
        )

In [ ]:
# Convert the date column to datetime
banco_inicial['data_venda'] = pd.to_datetime(
    banco_inicial['data_venda'],
    errors='coerce' # Convert invalid values to NaT
)

In [ ]:
banco_inicial.info()

### Handling Null and Missing Values

In [ ]:
banco_inicial[
    ['valor_unitario',
     'quantidade',
     'valor_total',
     'preco_custo',
     'saldo']
].isna().sum()

#### Missing Cost and Balance Values

In [ ]:
banco_inicial.groupby('grupo')[['preco_custo', 'saldo']].apply(
    lambda x: x.isna().sum()
)

Missing `preco_custo` and `saldo` values make profit analysis unreliable. The project therefore focuses on revenue, seasonality, and demand forecasting.

#### Missing `valor_total` and `valor_unitario` Values

Since `valor_total = valor_unitario * quantidade`, missing values can be inferred when the other two fields are available.

In [ ]:
# filling missing values of value_total and value_unitario
mask_total = (
    banco_inicial['valor_total'].isna()
    & banco_inicial['valor_unitario'].notna()
    & banco_inicial['quantidade'].notna()
)

banco_inicial.loc[mask_total, 'valor_total'] = (
    banco_inicial.loc[mask_total, 'valor_unitario']
    * banco_inicial.loc[mask_total, 'quantidade']
)

mask_unitario = (
    banco_inicial['valor_unitario'].isna()
    & banco_inicial['valor_total'].notna()
    & banco_inicial['quantidade'].notna()
    & (banco_inicial['quantidade'] > 0)
)

banco_inicial.loc[mask_unitario, 'valor_unitario'] = (
    banco_inicial.loc[mask_unitario, 'valor_total']
    / banco_inicial.loc[mask_unitario, 'quantidade']
)

In [ ]:
# Checking novamente the missing values após the filling
banco_inicial[
    ['valor_unitario',
     'quantidade',
     'valor_total',
     'preco_custo',
     'saldo']
].isna().sum()

Imputation reduced missing `valor_unitario` values from 1,882 to 794 (57.81% improvement) and missing `valor_total` values from 1,332 to 906 (31.98% improvement).

In [ ]:
banco_inicial = banco_inicial.dropna(
    subset=['valor_unitario', 'quantidade', 'valor_total'],
    how='all'
)

In [ ]:
# number of records restantes in the dataset of date
print(f"Number of records remaining in the dataset:: {banco_inicial.shape[0]}")

#### Records with `valor_total` Present but `valor_unitario` and `quantidade` Missing

In [ ]:
mask = (
    banco_inicial['valor_total'].notna()
    & banco_inicial['valor_unitario'].isna()
    & banco_inicial['quantidade'].isna()
)

mask.sum()

These records do not contain enough information for reliable analysis. They are removed because they are few and do not materially affect the dataset.

In [ ]:
banco_inicial = banco_inicial[~mask].copy()

#### Records with `quantidade` Missing but `valor_unitario` and `valor_total` Present

In [ ]:
mask_qtdna = (
    banco_inicial['valor_unitario'].notna()
    & banco_inicial['quantidade'].isna()
    & banco_inicial['valor_total'].notna()
)

mask_qtdna.sum()

In [ ]:
# Calculating a quantity a partir of the value_total and value_unitario
qtd_calc = (
    banco_inicial.loc[mask_qtdna, 'valor_total']
    / banco_inicial.loc[mask_qtdna, 'valor_unitario']
)
print(qtd_calc)

In [ ]:
# Checking quantos values calculados are inteiros
print((qtd_calc % 1 == 0).sum())

In [ ]:
banco_inicial = banco_inicial[~mask_qtdna].copy()

Fractional quantities are inconsistent with the way products are sold. They may reflect discounts or transaction-level adjustments, so those records are excluded instead of creating a noisy adjustment feature.

#### Records with `valor_unitario` Missing but `quantidade` and `valor_total` Present

In [ ]:
mask_vuna = (
    banco_inicial['valor_unitario'].isna()
    & banco_inicial['quantidade'].notna()
    & banco_inicial['valor_total'].notna()
)

mask_vuna.sum()

#### Records with `valor_unitario`, `quantidade`, and `valor_total` Missing

In [ ]:
mask_numna = (
    banco_inicial['valor_unitario'].isna()
    & banco_inicial['quantidade'].isna()
    & banco_inicial['valor_total'].isna()
)

mask_numna.sum()

#### Date Validation

In [ ]:
# number of records restantes in the dataset of date
print(f"Number of records remaining in the dataset:: {banco_inicial.shape[0]}")

In [ ]:
banco_inicial.info()

Records with missing sale dates cannot be used in time-series analysis and are removed.

In [ ]:
# number of records with sales records missing
mask_datna = (
    banco_inicial['data_venda'].isna()
)

mask_datna.sum()

In [ ]:
# Removing the records with dates missing
banco_inicial = banco_inicial[~mask_datna].copy()

#### Null or Negative Quantities

In [ ]:
# number of records with quantity lower or igual a zero
banco_inicial[banco_inicial['quantidade'] <= 0].shape[0]

In [ ]:
# number of records with values_unitarios lower or igual a zero
banco_inicial[banco_inicial['valor_unitario'] <= 0].shape[0]

In [ ]:
banco_inicial[banco_inicial['valor_unitario'] <= 0]

In [ ]:
# number of records with value_total lower or igual a zero
banco_inicial[banco_inicial['valor_total'] <= 0].shape[0]

In [ ]:
banco_inicial[banco_inicial['valor_total'] <= 0]

In [ ]:
# Removing Inconsistencies of quantity, value_unitario and value_total
banco_inicial.drop([3658, 5877, 7871], inplace=True)

In [ ]:
banco_inicial.reset_index(drop=True, inplace=True)

In [ ]:
banco_inicial.info()

In [ ]:
# Removing the records with missing values in quantity, value_unitario and value_total
banco_inicial.dropna(subset=['valor_unitario', 'quantidade', 'valor_total'], inplace=True)

In [ ]:
banco_inicial.info()

In [ ]:
# number of records restantes in the dataset of date
print(f"Number of records remaining in the dataset:: {banco_inicial.shape[0]}")

In [ ]:
banco_inicial.to_csv(
    "../data/interm/data_1.csv",
    index=False
)

---

## Initial Statistical Analysis

In [ ]:
banco_inicial.describe()

### Interpreting Descriptive Statistics

The cost and balance fields contain many missing and inconsistent values. The sales variables also have wide ranges because the dataset combines different products and sales units.

---

# Conclusion

The dataset is suitable for forecasting after the required cleaning steps. The next stages should determine which groups and revenue definition provide the most stable target for time-series modeling.